In [ ]:
# ============================================================
# CV ML ANALYSIS v2 - 10 FEATURES, FIVE VALIDATION SCHEMES
#
# Features: Potential, ScanRate_Index, Sweep_Direction (+ 7 material descriptors)
#
# Validation schemes
#   1. Holdout 80:20, single random split, per scan rate   (original paper's method)
#   2. Shuffled 5-fold CV within each curve (per scan rate)
#   3. Block (contiguous) 5-fold CV within each curve - no shuffling
#   4. Leave-one-scan-rate-out - an entire curve is held out
#   5. Combined (pooled) model, scan rate as input:
#        5a shuffled 5-fold   5b purged 5-fold
# Extras: feature ablation, nested tuning, conformal intervals,
#         leave-one-electrode-out, SHAP, sweep-direction check,
#         fold sizes and dataset sizes (for the reviewer's request)
# ============================================================

import os, sys, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              ExtraTreesRegressor, HistGradientBoostingRegressor,
                              StackingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import sklearn.base as skbase
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# ---- scikit-optimize: try to install automatically if missing ----
def _import_skopt():
    global BayesSearchCV, Real, Integer
    from skopt import BayesSearchCV
    from skopt.space import Real, Integer
try:
    _import_skopt(); BAYES_AVAILABLE = True
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-optimize"])
        _import_skopt(); BAYES_AVAILABLE = True
    except Exception:
        BAYES_AVAILABLE = False
print(f"BayesSearchCV available: {BAYES_AVAILABLE}")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available - skipping SHAP")

# ─────────────────────────────────────────────────────────
# SETTINGS
# ─────────────────────────────────────────────────────────
RUN_STANDARD_CV  = True    # shuffled 5-fold on pooled data, all 14 models (optimistic)
NESTED_TUNING    = True    # BayesSearchCV inside each purged fold (slow); needs skopt
TUNE_ITER        = 15
USE_DESCRIPTORS  = True    # False -> 3 features only (descriptors are constant per electrode)
SWEEP_WINDOW     = 21      # majority-vote window for sweep labels (raise to 51 if labels are still noisy)
ONLY_PER_CURVE   = False   # True -> run only schemes 1-3 (fast) and save ML_REPORT_percurve_only.xlsx
N_CHUNKS, N_SPLITS, BUFFER = 200, 5, 3
SEED = 42
np.random.seed(SEED)
OUT = "ULTIMATE_RESULTS"
os.makedirs(OUT, exist_ok=True)

try:
    from google.colab import files
    print("Colab mode - upload Excel files")
    uploaded = files.upload()
    file_names = list(uploaded.keys())
except Exception:
    file_names = ["Ag-TiO2-MnO2.xlsx", "Bi-TiO2-MnO2.xlsx",
                  "Pb-TiO2-MnO2.xlsx", "TiO2-MnO2.xlsx"]
    print(f"Local mode: {file_names}")

MATERIAL_DESCRIPTORS = {
    "Ag-TiO2-MnO2": {"Atomic_Number":47,"Atomic_Radius_pm":144,"Ionic_Radius_pm":126,
                     "Electronegativity":1.93,"Oxidation_State":1,"Valence_Electrons":1,"Work_Function_eV":4.26},
    "Bi-TiO2-MnO2": {"Atomic_Number":83,"Atomic_Radius_pm":143,"Ionic_Radius_pm":103,
                     "Electronegativity":2.02,"Oxidation_State":3,"Valence_Electrons":5,"Work_Function_eV":4.22},
    "Pb-TiO2-MnO2": {"Atomic_Number":82,"Atomic_Radius_pm":175,"Ionic_Radius_pm":119,
                     "Electronegativity":2.33,"Oxidation_State":2,"Valence_Electrons":4,"Work_Function_eV":4.25},
    "TiO2-MnO2":    {"Atomic_Number":0,"Atomic_Radius_pm":0,"Ionic_Radius_pm":0,
                     "Electronegativity":0,"Oxidation_State":0,"Valence_Electrons":0,"Work_Function_eV":0},
}
DESCRIPTOR_COLS = ["Atomic_Number","Atomic_Radius_pm","Ionic_Radius_pm",
                   "Electronegativity","Oxidation_State","Valence_Electrons","Work_Function_eV"]
BASE3 = ["Potential", "ScanRate_Index", "Sweep_Direction"]
FEATURES = BASE3 + (DESCRIPTOR_COLS if USE_DESCRIPTORS else [])   # Potential must be column 0

SCAN_PAIRS = [
    ("Potential_5", "5 mVsec-1",  "5 mV/s",  0),
    ("Potential_10","10 mVsec-1", "10 mV/s", 1),
    ("Potential_15","15 mVsec-1", "15 mV/s", 2),
    ("Potential_20","20 mVsec-1", "20 mV/s", 3),
    ("Potential_25","25mVsec-1",  "25 mV/s", 4),
    ("Potential_50","50 mVsec-1", "50 mV/s", 5),
]
SR_LABELS = {p[3]: p[2] for p in SCAN_PAIRS}
SCALE_MODELS = {"SVR","MLP","KNN (5)","KNN (7)","KNN (9)"}

# ─────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────
def to_subscript(text):
    return str(text).translate(str.maketrans("0123456789","₀₁₂₃₄₅₆₇₈₉"))

def get_colors(n=14):
    base = ["#e6194b","#3cb44b","#4363d8","#f58231","#911eb4","#42d4f4","#f032e6",
            "#bfef45","#469990","#dcbeff","#9A6324","#800000","#aaffc3","#808000"]
    return base[:n]

def add_sweep_direction(potential_series, window=None):
    """Forward (potential increasing) = 1, reverse = 0.
    Robust version: repeated potential values (ties) inherit the previous direction, and a
    centred majority vote over `window` points removes isolated sign flips caused by noise."""
    window = window or SWEEP_WINDOW
    pot = np.asarray(potential_series, dtype=float)
    if len(pot) < 3:
        return np.zeros(len(pot))
    step = np.sign(np.diff(pot))
    step = pd.Series(np.where(step == 0, np.nan, step)).ffill().bfill().fillna(0).values
    vote = pd.Series(step).rolling(window, center=True, min_periods=1).mean().values
    d = np.empty(len(pot))
    d[1:] = (vote > 0).astype(float)
    d[0] = d[1]
    return d

def add_sweep_direction_raw(potential_series):
    """Old naive rule, kept only to report how many labels the robust rule changed."""
    pot = np.asarray(potential_series, dtype=float)
    d = np.zeros(len(pot)); d[1:] = (np.diff(pot) > 0).astype(float); d[0] = d[1]
    return d

def build_models():
    base = [("histgb", HistGradientBoostingRegressor(random_state=SEED)),
            ("knn", KNeighborsRegressor(n_neighbors=7)),
            ("lgbm", LGBMRegressor(random_state=SEED, verbose=-1))]
    return {
        "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
        "Extra Trees":       ExtraTreesRegressor(n_estimators=200, random_state=SEED),
        "HistGB":            HistGradientBoostingRegressor(random_state=SEED),
        "SVR":               SVR(),
        "KNN (5)":           KNeighborsRegressor(n_neighbors=5),
        "KNN (7)":           KNeighborsRegressor(n_neighbors=7),
        "KNN (9)":           KNeighborsRegressor(n_neighbors=9),
        "MLP":               MLPRegressor(max_iter=2000, random_state=SEED),
        "Linear":            LinearRegression(),
        "XGBoost":           XGBRegressor(random_state=SEED, verbosity=0),
        "LightGBM":          LGBMRegressor(random_state=SEED, verbose=-1),
        "CatBoost":          CatBoostRegressor(random_state=SEED, verbose=0),
        "Stacking (Diverse)":StackingRegressor(estimators=base,
                                               final_estimator=Ridge(alpha=1.0), cv=5),
    }

# model factories used for the headline comparisons
MK = {"Extra Trees": lambda: ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
      "HistGB":      lambda: HistGradientBoostingRegressor(random_state=SEED)}

TUNE_FAILS = 0
def tune_histgb(X, y, n_iter=TUNE_ITER):
    """BayesSearchCV for HistGB. Falls back to defaults (and counts a failure) if it errors."""
    global TUNE_FAILS
    if not BAYES_AVAILABLE:
        TUNE_FAILS += 1
        return HistGradientBoostingRegressor(random_state=SEED)
    try:
        space = {"max_iter": Integer(100, 500), "max_leaf_nodes": Integer(15, 63),
                 "min_samples_leaf": Integer(10, 50), "l2_regularization": Real(0, 1.0),
                 "learning_rate": Real(0.01, 0.3, prior="log-uniform")}
        bs = BayesSearchCV(HistGradientBoostingRegressor(random_state=SEED), space,
                           n_iter=n_iter, cv=3, scoring="r2", n_jobs=-1, random_state=SEED)
        bs.fit(X, y)
        return bs.best_estimator_
    except Exception as e:
        TUNE_FAILS += 1
        print(f"      tuning failed ({e}); using defaults")
        return HistGradientBoostingRegressor(random_state=SEED)

def metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAE": mean_absolute_error(y_true, y_pred)}

# ─────────────────────────────────────────────────────────
# SPLITTERS
# ─────────────────────────────────────────────────────────
def purged_splits(potential, n_chunks=N_CHUNKS, n_splits=N_SPLITS,
                  buffer_chunks=BUFFER, seed=SEED):
    """(train_idx, test_idx) as indices into the original arrays.
    Potential axis -> contiguous bands (chunks); random bands form each test fold;
    `buffer_chunks` bands on each side of every test band are dropped from training."""
    n = len(potential)
    order = np.argsort(potential, kind="stable")
    bounds = np.linspace(0, n, n_chunks + 1).astype(int)
    chunk_of = np.empty(n, dtype=int)
    for i in range(n_chunks):
        chunk_of[order[bounds[i]:bounds[i + 1]]] = i
    rng = np.random.RandomState(seed)
    folds = np.array_split(rng.permutation(n_chunks), n_splits)
    for test_chunks in folds:
        test_set = set(test_chunks.tolist())
        purge = {c + s * b for c in test_set for b in range(1, buffer_chunks + 1) for s in (-1, 1)}
        train_set = set(range(n_chunks)) - test_set - purge
        te = np.where(np.isin(chunk_of, list(test_set)))[0]
        tr = np.where(np.isin(chunk_of, list(train_set)))[0]
        if len(te) and len(tr):
            yield tr, te

def loso_splits(sr):
    for s in np.unique(sr):
        yield np.where(sr != s)[0], np.where(sr == s)[0], int(s)

# ─────────────────────────────────────────────────────────
# GENERIC CV RUNNER
# ─────────────────────────────────────────────────────────
def _fit_predict(make_model, Xtr, ytr, Xte, needs_scale=False, tune=False):
    if needs_scale:
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    fm = tune_histgb(Xtr, ytr) if tune else make_model()
    fm.fit(Xtr, ytr)
    return fm.predict(Xte)

def run_cv(splits, X, y, make_model, needs_scale=False, tune=False, oof=None):
    """splits: iterable of (train_idx, test_idx[, label]). Returns per-fold DataFrame."""
    rows = []
    for k, sp in enumerate(splits):
        tr, te = sp[0], sp[1]
        label = sp[2] if len(sp) > 2 else k
        try:
            pred = _fit_predict(make_model, X[tr], y[tr], X[te], needs_scale, tune)
            if oof is not None:
                oof[te] = pred
            rows.append({"fold": label, "n_train": len(tr), "n_test": len(te),
                         **metrics(y[te], pred)})
        except Exception as e:
            print(f"      fold {label} failed: {e}")
    return pd.DataFrame(rows)

def conformal_coverage(X, y, alpha=0.10, n_blocks=100, cal_frac=0.2):
    """Split-conformal intervals (Extra Trees) under purged CV."""
    rng = np.random.RandomState(SEED)
    cov, wid = [], []
    for tr, te in purged_splits(X[:, 0]):
        o = tr[np.argsort(X[tr, 0], kind="stable")]
        blocks = np.array_split(o, n_blocks)
        cal_ids = set(rng.choice(n_blocks, int(n_blocks * cal_frac), replace=False).tolist())
        drop = {c + d for c in cal_ids for d in (-1, 1)}
        cal = np.concatenate([blocks[i] for i in sorted(cal_ids)])
        fit = np.concatenate([blocks[i] for i in range(n_blocks)
                              if i not in cal_ids and i not in drop])
        m = MK["Extra Trees"]().fit(X[fit], y[fit])
        scores = np.abs(y[cal] - m.predict(X[cal]))
        k = len(scores)
        q = np.quantile(scores, min(1.0, 0.9 * (k + 1) / k), method="higher")
        cov.append(np.mean(np.abs(y[te] - m.predict(X[te])) <= q))
        wid.append(2 * q)
    return float(np.mean(cov)), float(np.std(cov)), float(np.mean(wid))

def rf_oof_interval_coverage(X, y, percentile=90):
    cov, width = [], []
    lo_q, hi_q = (100 - percentile) / 2, 100 - (100 - percentile) / 2
    for tr, te in purged_splits(X[:, 0]):
        rf = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1).fit(X[tr], y[tr])
        tp = np.array([t.predict(X[te]) for t in rf.estimators_])
        lo, hi = np.percentile(tp, lo_q, axis=0), np.percentile(tp, hi_q, axis=0)
        cov.append(np.mean((y[te] >= lo) & (y[te] <= hi))); width.append(np.mean(hi - lo))
    return float(np.mean(cov)), float(np.mean(width))

# ─────────────────────────────────────────────────────────
# PER-CURVE SCHEMES (1, 2, 3): one model per scan-rate curve
# ─────────────────────────────────────────────────────────
PC_FEATURE_SETS = {"Potential only": ["Potential"],
                   "Potential + Sweep": ["Potential", "Sweep_Direction"]}

def per_curve_schemes(df, mat):
    rows, sizes = [], []
    for s in sorted(df["ScanRate_Index"].unique()):
        d = df[df["ScanRate_Index"] == s].reset_index(drop=True)   # original curve order kept
        y = d["Current"].values; n = len(d)
        idx = np.arange(n)
        h_tr, h_te = train_test_split(idx, test_size=0.2, random_state=SEED)
        schemes = {
            "1 Holdout 80:20":        [(h_tr, h_te)],
            "2 Shuffled 5-fold":      list(KFold(5, shuffle=True, random_state=SEED).split(idx)),
            "3 Block 5-fold":         list(KFold(5, shuffle=False).split(idx)),
        }
        for sch, sp in schemes.items():
            sizes.append({"Electrode": mat, "Scheme": sch, "ScanRate_Index": int(s),
                          "ScanRate": SR_LABELS[int(s)], "n_curve": n,
                          "n_train_mean": float(np.mean([len(a) for a, _ in sp])),
                          "n_test_mean": float(np.mean([len(b) for _, b in sp]))})
            for fs_name, cols in PC_FEATURE_SETS.items():
                X = d[cols].values
                for mname, mk in MK.items():
                    oof = np.full(n, np.nan)
                    r = run_cv(sp, X, y, mk, oof=oof)
                    if r.empty: continue
                    m_ = ~np.isnan(oof)
                    rows.append({"Electrode": mat, "Scheme": sch, "Model": mname,
                                 "Feature_Set": fs_name, "ScanRate_Index": int(s),
                                 "R2": r["R2"].mean(), "RMSE": r["RMSE"].mean(),
                                 # R2 on all out-of-fold predictions together (meaningful for block CV)
                                 "R2_pooled": r2_score(y[m_], oof[m_]),
                                 # RMSE relative to the SD of the whole curve
                                 "NRMSE": float(np.sqrt(np.mean((y[m_] - oof[m_]) ** 2)) / np.std(y))})
    return pd.DataFrame(rows), pd.DataFrame(sizes)

# ─────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────
summary_rows, percurve_rows, size_rows, fold_rows = [], [], [], []
std_rows, purged_rows, losro_rows, ablation_rows = [], [], [], []
unc_rows, shap_rows, sweep_rows, ds_rows = [], [], [], []
pooled_frames, oof_store, curve_store = [], {}, {}

ABLATION = {"Potential only":                 [0],
            "+ Sweep only":                    [0, 2],
            "+ ScanRate only":                 [0, 1],
            "+ ScanRate + Sweep (3 features)": [0, 1, 2]}
if USE_DESCRIPTORS:
    ABLATION["+ 7 Descriptors (10 features)"] = list(range(len(FEATURES)))

def add_summary(mat, scheme, model, feat, vals, note=""):
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0: return
    summary_rows.append({"Electrode": mat, "Scheme": scheme, "Model": model, "Feature_Set": feat,
                         "R2_Mean": round(float(vals.mean()), 4),
                         "R2_SD": round(float(vals.std(ddof=1)) if len(vals) > 1 else 0.0, 4),
                         "n_units": len(vals), "Unit": note})

print("\n" + "=" * 70)
print(f"ML ANALYSIS v2 - {len(FEATURES)} FEATURES, FIVE VALIDATION SCHEMES")
print("=" * 70)

for file in file_names:
    mat = file.replace(".xlsx", "")
    desc = MATERIAL_DESCRIPTORS.get(mat, MATERIAL_DESCRIPTORS["TiO2-MnO2"])
    df_raw = pd.read_excel(file)
    print(f"\n{'=' * 60}\n{mat}")

    parts = []
    for pot_col, cur_col, sr_label, sr_idx in SCAN_PAIRS:
        if pot_col not in df_raw.columns or cur_col not in df_raw.columns:
            continue
        t = df_raw[[pot_col, cur_col]].dropna().copy()
        t.columns = ["Potential", "Current"]
        t["ScanRate_Index"] = sr_idx
        t["Sweep_Direction"] = add_sweep_direction(t["Potential"])
        for k, v in desc.items():
            t[k] = v
        # sweep-direction sanity check
        flips = int(np.sum(np.diff(t["Sweep_Direction"].values) != 0))
        raw = add_sweep_direction_raw(t["Potential"])
        sweep_rows.append({"Electrode": mat, "ScanRate": sr_label, "n_points": len(t),
                           "Direction_Changes": flips,
                           "Direction_Changes_naive_rule": int(np.sum(np.diff(raw) != 0)),
                           "Tied_Potential_Steps": int(np.sum(np.diff(t["Potential"].values) == 0)),
                           "Labels_differing_from_naive": int(np.sum(raw != t["Sweep_Direction"].values))})
        ds_rows.append({"Electrode": mat, "ScanRate": sr_label, "ScanRate_Index": sr_idx,
                        "n_points": len(t)})
        parts.append(t)
    df = pd.concat(parts, ignore_index=True)
    df["Electrode"] = mat
    pooled_frames.append(df)
    curve_store[mat] = df

    X = df[FEATURES].values
    y = df["Current"].values
    sr = df["ScanRate_Index"].values
    print(f"   Points: {len(df)} | Features: {len(FEATURES)}")
    _sr = [r for r in sweep_rows if r["Electrode"] == mat]
    print(f"   Direction changes per curve (robust): {[r['Direction_Changes'] for r in _sr]}  (expect ~2)")
    print(f"   Direction changes per curve (naive) : {[r['Direction_Changes_naive_rule'] for r in _sr]}")
    print(f"   Tied potential steps per curve      : {[r['Tied_Potential_Steps'] for r in _sr]}")

    # ---------- schemes 1-3: per scan-rate curve ----------
    print("   Schemes 1-3 (per-curve models)...")
    pc, pc_sizes = per_curve_schemes(df, mat)
    percurve_rows.append(pc); size_rows.append(pc_sizes)
    for (sch, mname, fs), g in pc.groupby(["Scheme", "Model", "Feature_Set"]):
        add_summary(mat, sch, mname, fs, g["R2"].values, "scan-rate curves")
    for mname in MK:
        for sch in ["1 Holdout 80:20", "2 Shuffled 5-fold", "3 Block 5-fold"]:
            g = pc[(pc.Scheme == sch) & (pc.Model == mname) & (pc.Feature_Set == "Potential + Sweep")]
            if len(g):
                print(f"      {mname:<12} {sch:<20} R2(per fold)={g.R2.mean():.4f} +/- {g.R2.std(ddof=1):.4f} | "
                      f"R2(pooled)={g.R2_pooled.mean():.4f} | NRMSE={g.NRMSE.mean():.4f}")

    if ONLY_PER_CURVE:
        continue

    # ---------- scheme 4: leave-one-scan-rate-out ----------
    print("   Scheme 4 (leave-one-scan-rate-out)...")
    for mname, mk in MK.items():
        r = run_cv(loso_splits(sr), X, y, mk)
        for _, row in r.iterrows():
            losro_rows.append({"Electrode": mat, "Model": mname,
                               "Held_Out_ScanRate": SR_LABELS[int(row["fold"])],
                               "n_train": int(row["n_train"]), "n_test": int(row["n_test"]),
                               "R2": round(row["R2"], 4), "RMSE": round(row["RMSE"], 4)})
        add_summary(mat, "4 Leave-one-scan-rate-out", mname, "Pooled features", r["R2"].values, "held-out curves")
        if not r.empty:
            print(f"      {mname:<12} mean R2={r['R2'].mean():.4f} +/- {r['R2'].std(ddof=1):.4f} (min {r['R2'].min():.3f})")

    # ---------- scheme 5: combined model ----------
    print("   Scheme 5 (combined model: 5a shuffled, 5b purged)...")
    for mname, mk in MK.items():
        r5a = run_cv(KFold(5, shuffle=True, random_state=SEED).split(X), X, y, mk)
        add_summary(mat, "5a Pooled shuffled 5-fold", mname, "Pooled features", r5a["R2"].values, "folds")
        oof = np.full(len(y), np.nan)
        r5b = run_cv(purged_splits(X[:, 0]), X, y, mk, oof=oof)
        add_summary(mat, "5b Pooled purged 5-fold", mname, "Pooled features", r5b["R2"].values, "folds")
        if mname == "Extra Trees":
            oof_store[mat] = oof
            for _, row in r5b.iterrows():
                fold_rows.append({"Electrode": mat, "Scheme": "5b Pooled purged 5-fold",
                                  "Fold": int(row["fold"]), "n_train": int(row["n_train"]),
                                  "n_test": int(row["n_test"]), "n_total": len(y)})
            for _, row in r5a.iterrows():
                fold_rows.append({"Electrode": mat, "Scheme": "5a Pooled shuffled 5-fold",
                                  "Fold": int(row["fold"]), "n_train": int(row["n_train"]),
                                  "n_test": int(row["n_test"]), "n_total": len(y)})
        print(f"      {mname:<12} shuffled R2={r5a['R2'].mean():.4f} | purged R2={r5b['R2'].mean():.4f} +/- {r5b['R2'].std(ddof=1):.4f}")

    # ---------- standard CV for all 14 models (pooled) ----------
    if RUN_STANDARD_CV:
        print("   Standard shuffled 5-fold, all 14 models (optimistic)...")
        for name, m in build_models().items():
            r = run_cv(KFold(5, shuffle=True, random_state=SEED).split(X), X, y,
                       lambda m=m: skbase.clone(m), needs_scale=name in SCALE_MODELS)
            if not r.empty:
                std_rows.append({"Electrode": mat, "Model": name,
                                 "CV_R2_Mean": round(r["R2"].mean(), 4),
                                 "CV_R2_SD": round(r["R2"].std(ddof=1), 4),
                                 "CV_RMSE_Mean": round(r["RMSE"].mean(), 4),
                                 "CV_MAE_Mean": round(r["MAE"].mean(), 4)})

    # ---------- purged CV, model comparison ----------
    print("   Purged 5-fold, model comparison...")
    models = build_models()
    for name in ["Extra Trees", "Random Forest", "HistGB", "LightGBM", "Stacking (Diverse)"]:
        r = run_cv(purged_splits(X[:, 0]), X, y, lambda m=models[name]: skbase.clone(m),
                   needs_scale=name in SCALE_MODELS)
        if not r.empty:
            purged_rows.append({"Electrode": mat, "Model": name,
                                "Purged_R2_Mean": round(r["R2"].mean(), 4),
                                "Purged_R2_SD": round(r["R2"].std(ddof=1), 4),
                                "Purged_RMSE_Mean": round(r["RMSE"].mean(), 4),
                                "Purged_MAE_Mean": round(r["MAE"].mean(), 4)})
            print(f"      {name:<20} R2={r['R2'].mean():.4f} +/- {r['R2'].std(ddof=1):.4f}")
    if NESTED_TUNING and BAYES_AVAILABLE:
        TUNE_FAILS = 0
        r = run_cv(purged_splits(X[:, 0]), X, y, None, tune=True)
        if not r.empty and TUNE_FAILS == 0:
            purged_rows.append({"Electrode": mat, "Model": "HistGB (Tuned, nested)",
                                "Purged_R2_Mean": round(r["R2"].mean(), 4),
                                "Purged_R2_SD": round(r["R2"].std(ddof=1), 4),
                                "Purged_RMSE_Mean": round(r["RMSE"].mean(), 4),
                                "Purged_MAE_Mean": round(r["MAE"].mean(), 4)})
            print(f"      {'HistGB (Tuned, nested)':<20} R2={r['R2'].mean():.4f} +/- {r['R2'].std(ddof=1):.4f}")
        else:
            print("      nested tuning skipped/failed - not reported")

    # ---------- ablation ----------
    print("   Feature ablation (purged CV)...")
    for mname, mk in MK.items():
        for label, idx in ABLATION.items():
            r = run_cv(purged_splits(X[:, 0]), X[:, idx], y, mk)
            ablation_rows.append({"Electrode": mat, "Model": mname, "Feature_Set": label,
                                  "Purged_R2_Mean": round(r["R2"].mean(), 4),
                                  "Purged_R2_SD": round(r["R2"].std(ddof=1), 4)})
            print(f"      {mname:<12} {label:<34} R2={r['R2'].mean():.4f} +/- {r['R2'].std(ddof=1):.4f}")

    # ---------- uncertainty ----------
    cov, wid = rf_oof_interval_coverage(X, y, 90)
    ccov, cstd, cwid = conformal_coverage(X, y, 0.10)
    unc_rows.append({"Electrode": mat,
                     "RF_Coverage_Nominal90": round(cov, 4), "RF_Width": round(wid, 4),
                     "Conformal_ET_Coverage_Nominal90": round(ccov, 4),
                     "Conformal_Coverage_SD": round(cstd, 4), "Conformal_ET_Width": round(cwid, 4)})
    print(f"   90% intervals (held-out): RF coverage={cov:.3f} | conformal ET coverage={ccov:.3f}")

    # ---------- SHAP (Extra Trees) ----------
    if SHAP_AVAILABLE:
        try:
            et = MK["Extra Trees"]().fit(X, y)
            rs = np.random.RandomState(SEED)
            bg = X[rs.choice(len(X), min(100, len(X)), replace=False)]
            ex = X[rs.choice(len(X), min(300, len(X)), replace=False)]
            ms = np.abs(shap.Explainer(et.predict, bg)(ex).values).mean(axis=0)
            for rank, j in enumerate(np.argsort(-ms), 1):
                shap_rows.append({"Electrode": mat, "Feature": FEATURES[j],
                                  "SHAP_Importance": round(float(ms[j]), 6), "Rank": rank})
        except Exception as e:
            print(f"      SHAP error: {e}")

percurve_all = pd.concat(percurve_rows, ignore_index=True)
pc_pool = (percurve_all.groupby(["Electrode", "Scheme", "Model", "Feature_Set"])
           .agg(R2_perfold_mean=("R2", "mean"), R2_pooled_mean=("R2_pooled", "mean"),
                R2_pooled_sd=("R2_pooled", "std"), NRMSE_mean=("NRMSE", "mean"))
           .round(4).reset_index())
if ONLY_PER_CURVE:
    with pd.ExcelWriter(f"{OUT}/ML_REPORT_percurve_only.xlsx") as xw:
        pc_pool.to_excel(xw, sheet_name="Schemes1-3_Pooled_NRMSE", index=False)
        percurve_all.round(4).to_excel(xw, sheet_name="Per_Curve_Detail", index=False)
        pd.concat(size_rows, ignore_index=True).round(1).to_excel(xw, sheet_name="Fold_Sizes_PerCurve", index=False)
    print("\nSchemes 1-3, Potential + Sweep:")
    print(pc_pool[pc_pool.Feature_Set == "Potential + Sweep"].to_string(index=False))
    sys.exit(0)

# ─────────────────────────────────────────────────────────
# LEAVE-ONE-ELECTRODE-OUT (pooled; tests the descriptors)
# ─────────────────────────────────────────────────────────
print("\nPooled model, leave-one-electrode-out (HistGB)...")
pooled = pd.concat(pooled_frames, ignore_index=True)
loeo_rows = []
for held in pooled["Electrode"].unique():
    tr = pooled[pooled["Electrode"] != held]; te = pooled[pooled["Electrode"] == held]
    for label, cols in [("3 features (no descriptors)", BASE3),
                        ("10 features (with descriptors)", BASE3 + DESCRIPTOR_COLS)]:
        m = HistGradientBoostingRegressor(random_state=SEED).fit(tr[cols].values, tr["Current"].values)
        r = metrics(te["Current"].values, m.predict(te[cols].values))
        loeo_rows.append({"Held_Out_Electrode": held, "Feature_Set": label,
                          "R2": round(r["R2"], 4), "RMSE": round(r["RMSE"], 4)})
        print(f"   held out {held:<14} {label:<32} R2={r['R2']:.4f}")

# ─────────────────────────────────────────────────────────
# TABLES
# ─────────────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows)
percurve_df = pd.concat(percurve_rows, ignore_index=True)
sizes_df = pd.concat(size_rows, ignore_index=True)
std_df, purge_df = pd.DataFrame(std_rows), pd.DataFrame(purged_rows)
abl_df, unc_df, shap_df = pd.DataFrame(ablation_rows), pd.DataFrame(unc_rows), pd.DataFrame(shap_rows)
losro_df, loeo_df = pd.DataFrame(losro_rows), pd.DataFrame(loeo_rows)
fold_df, sweep_df, ds_df = pd.DataFrame(fold_rows), pd.DataFrame(sweep_rows), pd.DataFrame(ds_rows)
elecs = [f.replace(".xlsx", "") for f in file_names]

# ─────────────────────────────────────────────────────────
# FIGURES
# ─────────────────────────────────────────────────────────
print("\nGenerating figures...")

# Fig 12: predicted vs experimental (out-of-fold, purged CV, Extra Trees), per electrode
for mat in elecs:
    df = curve_store[mat]; oof = oof_store.get(mat)
    if oof is None: continue
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, s in zip(axes.ravel(), sorted(df["ScanRate_Index"].unique())):
        m = (df["ScanRate_Index"] == s).values
        yt, yp = df.loc[m, "Current"].values, oof[m]
        ok = ~np.isnan(yp)
        ax.scatter(yt[ok], yp[ok], s=6, alpha=0.5, color="#4363d8")
        lo, hi = min(yt.min(), np.nanmin(yp)), max(yt.max(), np.nanmax(yp))
        ax.plot([lo, hi], [lo, hi], "r--", lw=1)
        ax.set_title(f"{SR_LABELS[int(s)]}  (R² = {r2_score(yt[ok], yp[ok]):.3f})", fontsize=11, fontweight="bold")
        ax.set_xlabel("Experimental current (A)"); ax.set_ylabel("Predicted current (A)")
        ax.grid(True, alpha=0.4)
    fig.suptitle(f"Extra Trees - out-of-fold predictions (purged 5-fold CV) - {to_subscript(mat)}",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{OUT}/{mat}_scatter_oof.png", dpi=300, bbox_inches="tight"); plt.close()

# Fig 13: purged CV R2 of models, per electrode
for mat in elecs:
    d = purge_df[purge_df["Electrode"] == mat].sort_values("Purged_R2_Mean", ascending=False)
    if d.empty: continue
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(range(len(d)), d["Purged_R2_Mean"], yerr=d["Purged_R2_SD"], capsize=4,
           color=get_colors(len(d)), alpha=0.85)
    ax.set_xticks(range(len(d))); ax.set_xticklabels(d["Model"], rotation=30, ha="right")
    ax.set_ylabel(r"Purged 5-fold CV $R^2$ (mean $\pm$ SD)")
    ax.set_title(f"Purged CV R2 - {to_subscript(mat)}", fontweight="bold")
    ax.axhline(0, color="black", lw=0.8, ls="--"); ax.grid(True, axis="y", alpha=0.4)
    for xi, (m_, s_) in enumerate(zip(d["Purged_R2_Mean"], d["Purged_R2_SD"])):
        ax.text(xi, m_ + s_ + 0.01, f"{m_:.3f}", ha="center", fontsize=8)
    plt.tight_layout(); plt.savefig(f"{OUT}/{mat}_purged_cv_r2.png", dpi=300, bbox_inches="tight"); plt.close()

# Fig 14: ablation (Extra Trees)
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(elecs)); labels = list(ABLATION.keys()); w = 0.8 / len(labels)
cols = ["#e6194b", "#f58231", "#911eb4", "#3cb44b", "#4363d8"][:len(labels)]
a = abl_df[abl_df["Model"] == "Extra Trees"]
for k, (label, col) in enumerate(zip(labels, cols)):
    sub = a[a["Feature_Set"] == label].set_index("Electrode").reindex(elecs)
    ax.bar(x + (k - (len(labels) - 1) / 2) * w, sub["Purged_R2_Mean"], w, yerr=sub["Purged_R2_SD"],
           capsize=3, label=label, color=col, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([to_subscript(e) for e in elecs])
ax.set_ylabel(r"Purged CV $R^2$"); ax.set_ylim(min(0, a["Purged_R2_Mean"].min() - 0.1), 1.1)
ax.set_title("Feature Ablation under Purged 5-Fold CV (Extra Trees)", fontweight="bold")
ax.axhline(0, color="black", lw=0.8, ls="--"); ax.grid(True, axis="y", alpha=0.4); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/improvement_comparison.png", dpi=300, bbox_inches="tight"); plt.close()

# Fig 15: all five validation schemes (Extra Trees); schemes 1-3 use R2 of pooled out-of-fold predictions
pp = pc_pool[(pc_pool.Model == "Extra Trees") & (pc_pool.Feature_Set == "Potential + Sweep")]
ss = summary_df[(summary_df.Model == "Extra Trees") & (summary_df.Feature_Set == "Pooled features")]
spec = [("1 Holdout 80:20 (per curve)", pp, "1 Holdout 80:20", "R2_pooled_mean", "R2_pooled_sd"),
        ("2 Shuffled 5-fold (per curve)", pp, "2 Shuffled 5-fold", "R2_pooled_mean", "R2_pooled_sd"),
        ("3 Block 5-fold (per curve, extrapolation)", pp, "3 Block 5-fold", "R2_pooled_mean", "R2_pooled_sd"),
        ("4 Leave-one-scan-rate-out", ss, "4 Leave-one-scan-rate-out", "R2_Mean", "R2_SD"),
        ("5a Combined, shuffled 5-fold", ss, "5a Pooled shuffled 5-fold", "R2_Mean", "R2_SD"),
        ("5b Combined, purged 5-fold", ss, "5b Pooled purged 5-fold", "R2_Mean", "R2_SD")]
pal = ["#e6194b", "#f58231", "#bfef45", "#3cb44b", "#42d4f4", "#4363d8"]
fig, ax = plt.subplots(figsize=(14, 6)); w = 0.8 / len(spec); allv = []
for k, ((lab, df_, key, mc, sc), col) in enumerate(zip(spec, pal)):
    dd = df_[df_["Scheme"] == key].set_index("Electrode").reindex(elecs)
    allv += list(dd[mc].values)
    ax.bar(x + (k - (len(spec) - 1) / 2) * w, dd[mc], w, yerr=dd[sc].fillna(0), capsize=3, label=lab, color=col, alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels([to_subscript(e) for e in elecs])
ax.set_ylabel(r"$R^2$ (mean $\pm$ SD)"); ax.set_ylim(min(0, np.nanmin(allv) - 0.05), 1.1)
ax.set_title("Extra Trees under five validation schemes", fontweight="bold")
ax.axhline(0, color="black", lw=0.8, ls="--"); ax.grid(True, axis="y", alpha=0.4)
ax.legend(fontsize=9, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.07), frameon=False)
plt.tight_layout(); plt.savefig(f"{OUT}/validation_schemes_comparison.png", dpi=300, bbox_inches="tight"); plt.close()

# Standard vs purged (pooled)
if RUN_STANDARD_CV and not std_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6)); w = 0.35
    s = std_df[std_df["Model"] == "Extra Trees"].set_index("Electrode").reindex(elecs)
    p = purge_df[purge_df["Model"] == "Extra Trees"].set_index("Electrode").reindex(elecs)
    ax.bar(x - w/2, s["CV_R2_Mean"], w, yerr=s["CV_R2_SD"], capsize=4, label="Standard shuffled 5-fold", color="#4363d8", alpha=0.85)
    ax.bar(x + w/2, p["Purged_R2_Mean"], w, yerr=p["Purged_R2_SD"], capsize=4, label="Purged 5-fold", color="#f58231", alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels([to_subscript(e) for e in elecs]); ax.set_ylabel(r"$R^2$")
    ax.set_title("Standard vs Purged 5-Fold CV - Extra Trees", fontweight="bold")
    ax.axhline(0, color="black", lw=0.8, ls="--"); ax.grid(True, axis="y", alpha=0.4); ax.legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/purged_vs_standard_cv.png", dpi=300, bbox_inches="tight"); plt.close()

# SHAP
if SHAP_AVAILABLE and not shap_df.empty:
    fig, axes = plt.subplots(1, len(elecs), figsize=(5 * len(elecs), 7)); axes = np.atleast_1d(axes)
    for ax, mat in zip(axes, elecs):
        d = shap_df[shap_df["Electrode"] == mat].sort_values("SHAP_Importance")
        ax.barh(d["Feature"], d["SHAP_Importance"], color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(d))), alpha=0.85)
        ax.set_xlabel("Mean |SHAP value|"); ax.set_title(to_subscript(mat), fontweight="bold"); ax.grid(True, axis="x", alpha=0.4)
    fig.suptitle("SHAP Feature Importance - Extra Trees", fontweight="bold")
    plt.tight_layout(); plt.savefig(f"{OUT}/shap_importance.png", dpi=300, bbox_inches="tight"); plt.close()

# Sweep-direction check (first scan-rate curve of each electrode)
fig, axes = plt.subplots(2, len(elecs), figsize=(4.5 * len(elecs), 7), squeeze=False)
for j, mat in enumerate(elecs):
    d = curve_store[mat]; d0 = d[d["ScanRate_Index"] == d["ScanRate_Index"].min()].reset_index(drop=True)
    sc = axes[0, j].scatter(d0.index, d0["Potential"], c=d0["Sweep_Direction"], cmap="coolwarm", s=4)
    axes[0, j].set_title(to_subscript(mat), fontweight="bold"); axes[0, j].set_xlabel("Point index")
    axes[0, j].set_ylabel("Potential (V)")
    axes[1, j].scatter(d0["Potential"], d0["Current"], c=d0["Sweep_Direction"], cmap="coolwarm", s=4)
    axes[1, j].set_xlabel("Potential (V)"); axes[1, j].set_ylabel("Current (A)")
fig.suptitle("Sweep-direction labels (red = forward/anodic, blue = reverse/cathodic), first scan rate", fontweight="bold")
plt.tight_layout(); plt.savefig(f"{OUT}/sweep_direction_check.png", dpi=200, bbox_inches="tight"); plt.close()

# ─────────────────────────────────────────────────────────
# EXCEL REPORT
# ─────────────────────────────────────────────────────────
features_info = pd.DataFrame([
    ["Potential (V)","Experimental","Primary driver of current response"],
    ["ScanRate_Index","Experimental","Kinetic effects at different sweep speeds"],
    ["Sweep_Direction","Experimental","Anodic (forward) vs cathodic (reverse) branch; inferred from data order"],
    ["Atomic_Number","Atomic","Element identity / periodic trend"],
    ["Atomic_Radius_pm","Atomic","Lattice distortion on doping"],
    ["Ionic_Radius_pm","Atomic","Charge distribution and ion mobility"],
    ["Electronegativity","Electronic","Electron density and bond polarity"],
    ["Oxidation_State","Electronic","Charge carrier concentration"],
    ["Valence_Electrons","Electronic","Electron donation capacity"],
    ["Work_Function_eV","Electronic","Charge transfer / electron emission"],
], columns=["Feature","Category","Justification"])

with pd.ExcelWriter(f"{OUT}/ML_REPORT_v2.xlsx") as xw:
    summary_df.to_excel(xw, sheet_name="Validation_Summary", index=False)
    pc_pool.to_excel(xw, sheet_name="Schemes1-3_Pooled_NRMSE", index=False)
    percurve_df.round(4).to_excel(xw, sheet_name="Per_Curve_Detail_Schemes1-3", index=False)
    losro_df.to_excel(xw, sheet_name="Scheme4_Leave1ScanRateOut", index=False)
    purge_df.to_excel(xw, sheet_name="Purged_5Fold_Models", index=False)
    abl_df.to_excel(xw, sheet_name="Feature_Ablation", index=False)
    unc_df.to_excel(xw, sheet_name="Uncertainty", index=False)
    loeo_df.to_excel(xw, sheet_name="Leave1Electrode_Out", index=False)
    fold_df.to_excel(xw, sheet_name="Fold_Sizes_Pooled", index=False)
    sizes_df.round(1).to_excel(xw, sheet_name="Fold_Sizes_PerCurve", index=False)
    ds_df.to_excel(xw, sheet_name="Dataset_Sizes", index=False)
    sweep_df.to_excel(xw, sheet_name="Sweep_Direction_Check", index=False)
    if not std_df.empty: std_df.to_excel(xw, sheet_name="Standard_5Fold_14models", index=False)
    if not shap_df.empty: shap_df.to_excel(xw, sheet_name="SHAP_Importance", index=False)
    features_info.to_excel(xw, sheet_name="Feature_List", index=False)

print("\n" + "=" * 70)
print(f"DONE | features: {len(FEATURES)} | seed: {SEED}")
print("\nVALIDATION SUMMARY (Extra Trees, R2 mean +/- SD):")
print(summary_df[summary_df.Model == "Extra Trees"].to_string(index=False))
print("\nPurged CV, model comparison:")
print(purge_df.to_string(index=False))
print(f"\nOutputs in {OUT}/")
print("=" * 70)